# Data Cleaning — Student Performance Dataset
**Author:** Minhaj Uddin Farhad
**Track:** Data Analytics (OIBSIP)
**Task:** Level 1 — Cleaning Data

**Objective:** Take a deliberately messy dataset and systematically transform it into a clean, analysis-ready dataset. Document every decision.

## Step 1: Load Data and Produce a Data Quality Report

We load the raw CSV and first inspect it: null counts, duplicate rows, data type issues, and any obvious value anomalies. This becomes our "before" snapshot, which we'll compare against the cleaned version later.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

# Note: this file isn't standard UTF-8 (it has a special character in a name),
# so we read it with latin1 encoding to avoid a UnicodeDecodeError.
df = pd.read_csv('bi.csv', encoding='latin1')

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

Shape: 77 rows, 11 columns


,fNAME,lNAME,Age,gender,country,residence,entryEXAM,prevEducation,studyHOURS,Python,DB
0,Christina,Binger,44,Female,Norway,Private,72,Masters,158,59.0,55
1,Alex,Walekhwa,60,M,Kenya,Private,79,Diploma,150,60.0,75
2,Philip,Leo,25,Male,Uganda,Sognsvann,55,HighSchool,130,74.0,50
3,Shoni,Hlongwane,22,F,Rsa,Sognsvann,40,High School,120,NaN,44
4,Maria,Kedibone,23,Female,South Africa,Sognsvann,65,High School,122,91.0,80
5,Hannah,Hansen,25,female,Norge,BI Residence,66,High School,130,88.0,59
6,Ole,Johansen,27,Male,Norway,BI-Residence,90,Bachelors,156,80.0,91
7,Lars,Olsen,29,Male,norway,BIResidence,89,Barrrchelors,160,85.0,60
8,Bjørn,Larsen,31,Male,Norway,BI Residence,88,Bachelors,156,80.0,89
9,Sofie,Jensen,33,Female,Denmark,BI_Residence,85,Bachelors,160,83.0,90


In [2]:
# --- DATA QUALITY REPORT ---

print("1. Null values per column:")
print(df.isnull().sum())
print()

print("2. Duplicate rows:", df.duplicated().sum())
print()

print("3. Data types:")
print(df.dtypes)
print()

print("4. Unique value counts per categorical column (flags inconsistent formatting):")
for col in ['gender', 'country', 'residence', 'prevEducation']:
    print(f"  {col}: {df[col].nunique()} unique values -> {sorted(df[col].unique())}")


1. Null values per column:
fNAME            0
lNAME            0
Age              0
gender           0
country          0
residence        0
entryEXAM        0
prevEducation    0
studyHOURS       0
Python           2
DB               0
dtype: int64

2. Duplicate rows: 0

3. Data types:
fNAME                str
lNAME                str
Age                int64
gender               str
country              str
residence            str
entryEXAM          int64
prevEducation        str
studyHOURS         int64
Python           float64
DB                 int64
dtype: object

4. Unique value counts per categorical column (flags inconsistent formatting):
  gender: 6 unique values -> ['F', 'Female', 'M', 'Male', 'female', 'male']
  country: 16 unique values -> ['Denmark', 'France', 'Germany', 'Italy', 'Kenya', 'Netherlands', 'Nigeria', 'Norge', 'Norway', 'Rsa', 'Somali', 'South Africa', 'Spain', 'UK', 'Uganda', 'norway']
  residence: 6 unique values -> ['BI Residence', 'BI-Residence', 'BIResid

**Data Quality Report Summary:** *(write your own take on this)*

- No duplicate rows.
- `Python` column has 2 missing values.
- Four categorical columns have inconsistent formatting that inflates their unique-value count beyond the real number of categories:
  - `gender`: 6 variants for what should be 2 categories (Male/Female)
  - `country`: 16 variants, with some being case differences (`Norway`/`norway`/`Norge`) and some being the same country under different names (`Rsa` vs `South Africa`)
  - `residence`: 6 variants for what should be 3 categories (spacing/casing/punctuation differences in "BI Residence")
  - `prevEducation`: 10 variants for what should be 5 categories (casing differences and typos like `Barrrchelors`, `Diplomaaa`)
- Numeric columns (Age, entryEXAM, studyHOURS, Python, DB) all look like reasonable ranges at a glance — we'll check for outliers formally in a later step.

## Step 2: Save a Snapshot of the Raw Data (for Before/After Comparison)

Before we start changing anything, we save the "before" stats so we can build a proper before-vs-after summary table at the end.

In [3]:
before_stats = {
    'null_count': df.isnull().sum().sum(),
    'duplicate_count': df.duplicated().sum(),
    'row_count': df.shape[0]
}
print(before_stats)

{'null_count': np.int64(2), 'duplicate_count': np.int64(0), 'row_count': 77}


## Step 3: Missing Data Handling

Only `Python` has missing values (2 rows). Since this is a numeric score column and only ~2.6% of rows are affected, we impute with the **median** rather than dropping rows (dropping would lose the rest of that student's data unnecessarily). Median is preferred over mean here because it's less sensitive to outliers in score data.

In [4]:
print("Missing Python scores before imputation:")
print(df[df['Python'].isnull()][['fNAME', 'lNAME', 'Python']])
print()

python_median = df['Python'].median()
print(f"Median Python score: {python_median}")

df['Python'] = df['Python'].fillna(python_median)

print()
print("Missing values remaining:", df['Python'].isnull().sum())

Missing Python scores before imputation:
    fNAME       lNAME  Python
3   Shoni   Hlongwane     NaN
33  Frank  Abrahamsen     NaN

Median Python score: 81.0

Missing values remaining: 0


## Step 4: Duplicate Removal

We already saw 0 duplicate rows in the data quality report, but we run `drop_duplicates()` anyway as a safety step and confirm the count removed.

In [5]:
rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]

print(f"Rows before: {rows_before}, Rows after: {rows_after}, Duplicates removed: {rows_before - rows_after}")

Rows before: 77, Rows after: 77, Duplicates removed: 0


## Step 5: Standardization

Now we fix the inconsistent categorical formatting we spotted in the data quality report. We do this one column at a time, printing before/after so each fix is verifiable.

In [6]:
# --- gender ---
print("Before:", sorted(df['gender'].unique()))

gender_map = {
    'Female': 'Female', 'female': 'Female', 'F': 'Female',
    'Male': 'Male', 'male': 'Male', 'M': 'Male'
}
df['gender'] = df['gender'].map(gender_map)

print("After:", sorted(df['gender'].unique()))

Before: ['F', 'Female', 'M', 'Male', 'female', 'male']
After: ['Female', 'Male']


In [7]:
# --- country ---
print("Before:", sorted(df['country'].unique()))

# Fix casing first (title case), then merge known synonyms
df['country'] = df['country'].str.strip().str.title()
df['country'] = df['country'].replace({'Rsa': 'South Africa', 'Norge': 'Norway', 'Uk': 'UK'})

print("After:", sorted(df['country'].unique()))

Before: ['Denmark', 'France', 'Germany', 'Italy', 'Kenya', 'Netherlands', 'Nigeria', 'Norge', 'Norway', 'Rsa', 'Somali', 'South Africa', 'Spain', 'UK', 'Uganda', 'norway']
After: ['Denmark', 'France', 'Germany', 'Italy', 'Kenya', 'Netherlands', 'Nigeria', 'Norway', 'Somali', 'South Africa', 'Spain', 'UK', 'Uganda']


In [8]:
# --- residence ---
print("Before:", sorted(df['residence'].unique()))

# All the "BI Residence" variants (different spacing/punctuation) become one consistent label
residence_map = {
    'BI Residence': 'BI Residence',
    'BI-Residence': 'BI Residence',
    'BIResidence': 'BI Residence',
    'BI_Residence': 'BI Residence',
    'Private': 'Private',
    'Sognsvann': 'Sognsvann'
}
df['residence'] = df['residence'].map(residence_map)

print("After:", sorted(df['residence'].unique()))

Before: ['BI Residence', 'BI-Residence', 'BIResidence', 'BI_Residence', 'Private', 'Sognsvann']
After: ['BI Residence', 'Private', 'Sognsvann']


In [9]:
# --- prevEducation ---
print("Before:", sorted(df['prevEducation'].unique()))

# Fix casing, then fix specific typos
df['prevEducation'] = df['prevEducation'].str.strip().str.title()
prevEd_map = {
    'Highschool': 'High School',
    'High School': 'High School',
    'Barrrchelors': 'Bachelors',
    'Diplomaaa': 'Diploma',
    'Diploma': 'Diploma',
    'Bachelors': 'Bachelors',
    'Masters': 'Masters',
    'Doctorate': 'Doctorate'
}
df['prevEducation'] = df['prevEducation'].replace(prevEd_map)

print("After:", sorted(df['prevEducation'].unique()))

Before: ['Bachelors', 'Barrrchelors', 'DIPLOMA', 'Diploma', 'Diplomaaa', 'Doctorate', 'High School', 'HighSchool', 'Masters', 'diploma']
After: ['Bachelors', 'Diploma', 'Doctorate', 'High School', 'Masters']


**Decisions made:** *(write your own take)*

- `gender`: mapped all 6 variants down to 2 standard values, `Male` and `Female`.
- `country`: applied title-case formatting, then manually merged known synonyms (`Rsa` → `South Africa`, `Norge` → `Norway`) since these refer to the same country under different names — this required domain knowledge, not just formatting rules.
- `residence`: merged 4 differently-punctuated versions of "BI Residence" into one consistent label.
- `prevEducation`: applied title-case, then fixed two data-entry typos (`Barrrchelors` → `Bachelors`, `Diplomaaa` → `Diploma`) that title-casing alone wouldn't catch.

## Step 6: Outlier Detection (IQR Method)

We check each numeric column for outliers using the IQR (Interquartile Range) method: any value below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` is flagged as an outlier. For each column, we decide whether to cap, remove, or retain the outliers, and document why.

In [10]:
numeric_cols = ['Age', 'entryEXAM', 'studyHOURS', 'Python', 'DB']

def find_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)], lower, upper

for col in numeric_cols:
    outliers, lower, upper = find_outliers_iqr(df[col])
    print(f"{col}: valid range [{lower:.1f}, {upper:.1f}] -> {len(outliers)} outlier(s)")
    if len(outliers) > 0:
        print(df.loc[outliers.index, ['fNAME', 'lNAME', col]])
    print()

Age: valid range [4.5, 64.5] -> 2 outlier(s)
      fNAME    lNAME  Age
32    Perry  Rønning   71
55  Chinedu   Okafor   69

entryEXAM: valid range [37.5, 121.5] -> 3 outlier(s)
       fNAME    lNAME  entryEXAM
32     Perry  Rønning         30
53   Chinedu  Morison         28
76  Mohammed    Salim         35

studyHOURS: valid range [123.0, 179.0] -> 7 outlier(s)
      fNAME      lNAME  studyHOURS
3     Shoni  Hlongwane         120
4     Maria   Kedibone         122
20     Prof  Birkeland         116
21    Hanna    Isaksen         114
32    Perry    Rønning         120
49     Thea    Knutsen         120
53  Chinedu    Morison         120

Python: valid range [52.5, 104.5] -> 6 outlier(s)
      fNAME          lNAME  Python
20     Prof      Birkeland    33.0
21    Hanna        Isaksen    30.0
32    Perry        Rønning    31.0
48    Jenny  Kristoffersen    48.0
49     Thea        Knutsen    45.0
53  Chinedu        Morison    15.0

DB: valid range [15.5, 123.5] -> 0 outlier(s)



**Outlier decisions:** *(rewrite this in your own words for the video/README)*

Actual outliers found by the IQR method:
- `Age`: 2 outliers (ages 71 and 69) — plausible for mature/executive students at a business school. **Decision: retain.**
- `entryEXAM`: 3 outliers (scores 28-35, on the low end) — these are unusually low but still valid exam scores (not negative, not above the max). **Decision: retain** — they represent genuinely weak performers, which is useful signal, not an error.
- `studyHOURS`: 7 outliers, all on the low end (114-122 hours vs. the rest clustering near 150-160) — still realistic study-hour values. **Decision: retain.**
- `Python`: 6 outliers, all low scores (15-48) — again, valid score range, just weak performance. **Decision: retain.**
- `DB`: 0 outliers found.

**Overall decision: retain all flagged values.** None of them fall outside a logically valid range for their column (no negative ages, no scores above 100, etc.) — they represent real variation in the data (some students genuinely performed worse or are older), not data-entry errors. Removing or capping them would discard real signal that could matter for later analysis (e.g., understanding what predicts low performance).

## Step 7: Data Type Correction

We confirm every column has the correct dtype: names/categories as strings, scores/age as numeric. This dataset doesn't have date or ID columns, so the main check is making sure our cleaned categorical columns are consistent object/string types and numeric columns are int/float.

In [11]:
print("Data types after cleaning:")
print(df.dtypes)
print()

# Explicitly cast categorical columns to string type (in case any became mixed type)
categorical_cols = ['fNAME', 'lNAME', 'gender', 'country', 'residence', 'prevEducation']
for col in categorical_cols:
    df[col] = df[col].astype(str)

# Numeric columns should be int/float - confirm no leftover object dtype
numeric_check_cols = ['Age', 'entryEXAM', 'studyHOURS', 'Python', 'DB']
for col in numeric_check_cols:
    df[col] = pd.to_numeric(df[col])

print("Confirmed dtypes after correction:")
print(df.dtypes)

Data types after cleaning:
fNAME                str
lNAME                str
Age                int64
gender               str
country              str
residence            str
entryEXAM          int64
prevEducation        str
studyHOURS         int64
Python           float64
DB                 int64
dtype: object

Confirmed dtypes after correction:
fNAME                str
lNAME                str
Age                int64
gender               str
country              str
residence            str
entryEXAM          int64
prevEducation        str
studyHOURS         int64
Python           float64
DB                 int64
dtype: object


## Step 8: Before vs. After Summary Table

A single table comparing key data-quality metrics before and after cleaning, so the impact of our work is clear at a glance.

In [12]:
after_stats = {
    'null_count': df.isnull().sum().sum(),
    'duplicate_count': df.duplicated().sum(),
    'row_count': df.shape[0]
}

# Count of categorical columns that had inconsistent formatting, before vs after
categorical_variants_before = {'gender': 6, 'country': 16, 'residence': 6, 'prevEducation': 10}
categorical_variants_after = {
    'gender': df['gender'].nunique(),
    'country': df['country'].nunique(),
    'residence': df['residence'].nunique(),
    'prevEducation': df['prevEducation'].nunique()
}

summary = pd.DataFrame({
    'Metric': ['Null values (total)', 'Duplicate rows', 'Row count',
               'gender - unique values', 'country - unique values',
               'residence - unique values', 'prevEducation - unique values',
               'All dtypes correct'],
    'Before': [before_stats['null_count'], before_stats['duplicate_count'], before_stats['row_count'],
               categorical_variants_before['gender'], categorical_variants_before['country'],
               categorical_variants_before['residence'], categorical_variants_before['prevEducation'],
               'No (mixed formatting in categoricals)'],
    'After': [after_stats['null_count'], after_stats['duplicate_count'], after_stats['row_count'],
              categorical_variants_after['gender'], categorical_variants_after['country'],
              categorical_variants_after['residence'], categorical_variants_after['prevEducation'],
              'Yes']
})

summary

,Metric,Before,After
0,Null values (total),2,0
1,Duplicate rows,0,0
2,Row count,77,77
3,gender - unique values,6,2
4,country - unique values,16,13
5,residence - unique values,6,3
6,prevEducation - unique values,10,5
7,All dtypes correct,No (mixed formatting in categoricals),Yes


## Step 9: Save the Cleaned Dataset

Finally, we export the cleaned dataframe to a new CSV file, keeping the original raw file untouched.

In [13]:
df.to_csv('bi_cleaned.csv', index=False)
print("Saved cleaned dataset as 'bi_cleaned.csv'")
print(f"Final shape: {df.shape}")
df.head(10)

Saved cleaned dataset as 'bi_cleaned.csv'
Final shape: (77, 11)


,fNAME,lNAME,Age,gender,country,residence,entryEXAM,prevEducation,studyHOURS,Python,DB
0,Christina,Binger,44,Female,Norway,Private,72,Masters,158,59.0,55
1,Alex,Walekhwa,60,Male,Kenya,Private,79,Diploma,150,60.0,75
2,Philip,Leo,25,Male,Uganda,Sognsvann,55,High School,130,74.0,50
3,Shoni,Hlongwane,22,Female,South Africa,Sognsvann,40,High School,120,81.0,44
4,Maria,Kedibone,23,Female,South Africa,Sognsvann,65,High School,122,91.0,80
5,Hannah,Hansen,25,Female,Norway,BI Residence,66,High School,130,88.0,59
6,Ole,Johansen,27,Male,Norway,BI Residence,90,Bachelors,156,80.0,91
7,Lars,Olsen,29,Male,Norway,BI Residence,89,Bachelors,160,85.0,60
8,Bjørn,Larsen,31,Male,Norway,BI Residence,88,Bachelors,156,80.0,89
9,Sofie,Jensen,33,Female,Denmark,BI Residence,85,Bachelors,160,83.0,90


## Conclusion

*(write your own 2-3 sentence wrap-up for the video/README)*

Starting from a 77-row student dataset with inconsistent gender/country/residence/education labels and 2 missing Python scores, we produced a clean, analysis-ready dataset: all categorical fields now use a single consistent format, missing values are imputed, and every cleaning decision (including retaining rather than removing flagged outliers) is documented and justified above.